In [ ]:
import os

os.environ['KAGGLE_API_TOKEN'] = "Kaggle API"


In [2]:
!kaggle competitions download -c super-ai-engineer-ss-6-thai-language-image-captioning

!ls

!unzip super-ai-engineer-ss-6-thai-language-image-captioning.zip -d super-ai-engineer-ss-6-thai-language-image-captioning

!ls super-ai-engineer-ss-6-thai-language-image-captioning

เอาต์พุตของการสตรีมมีการตัดเหลือเพียง 5000 บรรทัดสุดท้าย
  inflating: super-ai-engineer-ss-6-thai-language-image-captioning/train/train/travel/15349.jpg  
  inflating: super-ai-engineer-ss-6-thai-language-image-captioning/train/train/travel/15350.jpg  
  inflating: super-ai-engineer-ss-6-thai-language-image-captioning/train/train/travel/15351.jpg  
  inflating: super-ai-engineer-ss-6-thai-language-image-captioning/train/train/travel/15352.jpg  
  inflating: super-ai-engineer-ss-6-thai-language-image-captioning/train/train/travel/15353.jpg  
  inflating: super-ai-engineer-ss-6-thai-language-image-captioning/train/train/travel/15354.jpg  
  inflating: super-ai-engineer-ss-6-thai-language-image-captioning/train/train/travel/15355.jpg  
  inflating: super-ai-engineer-ss-6-thai-language-image-captioning/train/train/travel/15356.jpg  
  inflating: super-ai-engineer-ss-6-thai-language-image-captioning/train/train/travel/15357.jpg  
  inflating: super-ai-engineer-ss-6-thai-language-image-capti

In [3]:
!pip install -q transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 653.2 kB/s eta 0:00:00


In [4]:
from google.colab import drive
import os

drive.mount('/content/drive')

base_path = '/content/super-ai-engineer-ss-6-thai-language-image-captioning'
drive_coco_path = '/content/drive/MyDrive/MSCOCO_Dataset'
local_coco_dir = os.path.join(base_path, 'coco')

!mkdir -p {local_coco_dir}

!unzip -q {drive_coco_path}/train2017.zip -d {local_coco_dir}
!unzip -q {drive_coco_path}/val2017.zip -d {local_coco_dir}
print("Done")

Mounted at /content/drive
Done


In [ ]:
import torch
from PIL import Image
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, XLMRobertaTokenizer

checkpoint_path = '/content/drive/MyDrive/caption_model_checkpoints/checkpoint-16810'

encoder_id = "google/vit-base-patch16-384"
decoder_id = "airesearch/wangchanberta-base-att-spm-uncased"

image_processor = ViTImageProcessor.from_pretrained(encoder_id)
tokenizer = XLMRobertaTokenizer.from_pretrained(decoder_id)

model = VisionEncoderDecoderModel.from_pretrained(checkpoint_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

In [5]:
import os
import json
import torch
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from transformers import (
    VisionEncoderDecoderModel,
    ViTImageProcessor,
    XLMRobertaTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator,
    GenerationConfig
)

base_path = '/content/super-ai-engineer-ss-6-thai-language-image-captioning'
train_json_path = f'{base_path}/capgen_v1.0_train.json'
val_json_path = f'{base_path}/capgen_v1.0_val.json'
train_img_dir = f'{base_path}/train/train'
val_img_dir = f'{base_path}/val/val'

encoder_id = "google/vit-base-patch16-384"
decoder_id = "airesearch/wangchanberta-base-att-spm-uncased"

image_processor = ViTImageProcessor.from_pretrained(encoder_id)
tokenizer = XLMRobertaTokenizer.from_pretrained(decoder_id)

tokenizer.bos_token = tokenizer.cls_token
tokenizer.eos_token = tokenizer.sep_token

model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(encoder_id, decoder_id)

model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

model.config.decoder.pad_token_id = tokenizer.pad_token_id
model.config.decoder.bos_token_id = tokenizer.bos_token_id
model.config.decoder.eos_token_id = tokenizer.eos_token_id
model.config.vocab_size = model.config.decoder.vocab_size

generation_config = GenerationConfig(
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    max_length=64,
    early_stopping=True,
    num_beams=8,
    length_penalty=1.1,
    no_repeat_ngram_size=3,
    decoder_start_token_id=tokenizer.bos_token_id
)

model.generation_config = generation_config

for key in ['max_length', 'early_stopping', 'num_beams', 'length_penalty', 'no_repeat_ngram_size']:
    if hasattr(model.config, key):
        delattr(model.config, key)

class ThaiCaptionDataset(Dataset):
    def __init__(self, json_path, img_base_dir, processor, tokenizer, max_length=50, is_train=False):
        self.processor = processor
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []
        self.is_train = is_train

        self.transform = transforms.Compose([
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        ])

        self.corrections = {
            "สีน้ตาล": "สีน้ำตาล", "กลัง": "กำลัง", "สีด": "สีดำ",
            "แม่น้": "แม่น้ำ", "จนวน": "จำนวน", "ทท่า": "ทำท่า",
            "กแพง": "กำแพง", "น้ใส": "น้ำใส", "น้จิ้ม": "น้ำจิ้ม",
            "ถูกนมา": "ถูกนำมา", "นมาวาง": "นำมาวาง", "ในน้": "ในน้ำ",
            "ผิวน้": "ผิวน้ำ", "ริมน้": "ริมน้ำ", "น้ทะเล": "น้ำทะเล",
            "น้ตก": "น้ำตก", "น้พุ": "น้ำพุ", "น้แข็ง": "น้ำแข็ง",
            "อาบน้": "อาบน้ำ", "ว่ายน้": "ว่ายน้ำ", "ดน้": "ดำน้ำ",
            "ใต้น้": "ใต้น้ำ", "แก้วน้": "แก้วน้ำ", "ขวดน้": "ขวดน้ำ",
            "ทอาหาร": "ทำอาหาร", "จดจ": "จดจำ", "ประจ": "ประจำ",
            "บ่อน้": "บ่อน้ำ"
        }

        self.toxic_words = [
            "ฯลฯ", "ทำท่าทางยืน", "และมีก้อนเมฆ", "อาศัยอยู่ใน",
            "ภาพถ่ายระยะใกล้", "ภาพของนกที่กำลังยืนอยู่ในสวนสัตว์"
        ]

        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        for img_key, captions in data.items():
            filename = img_key.split('/')[-1]
            path_coco = os.path.join(base_path, img_key)
            category = img_key.split('/')[-2] if len(img_key.split('/')) > 1 else ""
            path_ipu = os.path.join(img_base_dir, category, filename)
            path_ipu_fallback = os.path.join(img_base_dir, filename)

            final_path = None
            if os.path.exists(path_coco): final_path = path_coco
            elif os.path.exists(path_ipu): final_path = path_ipu
            elif os.path.exists(path_ipu_fallback): final_path = path_ipu_fallback

            if final_path:
                for cap in captions:
                    if any(toxic in cap for toxic in self.toxic_words): continue
                    clean_cap = cap
                    for wrong, right in self.corrections.items():
                        clean_cap = clean_cap.replace(wrong, right)
                    self.samples.append({"image_path": final_path, "caption": clean_cap})


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item["image_path"]).convert("RGB")

        if self.is_train:
            image = self.transform(image)

        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()

        tokenized = self.tokenizer(
            item["caption"],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        input_ids = tokenized.input_ids.squeeze()
        attention_mask = tokenized.attention_mask.squeeze()

        decoder_input_ids = torch.full_like(input_ids, self.tokenizer.pad_token_id)
        decoder_input_ids[1:] = input_ids[:-1].clone()
        decoder_input_ids[0] = self.tokenizer.bos_token_id

        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": pixel_values,
            "decoder_input_ids": decoder_input_ids,
            "decoder_attention_mask": attention_mask,
            "labels": labels
        }

train_dataset = ThaiCaptionDataset(train_json_path, train_img_dir, image_processor, tokenizer, is_train=True)
val_dataset = ThaiCaptionDataset(val_json_path, val_img_dir, image_processor, tokenizer, is_train=False)


training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/caption_model_sota",
    per_device_train_batch_size=32,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    label_smoothing_factor=0.1,
    bf16=True,
    remove_unused_columns=False,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=1000,
    save_steps=1000,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    save_total_limit=2,
    logging_steps=200,
    predict_with_generate=False,
    report_to="none",
    dataloader_num_workers=4
)

trainer = Seq2SeqTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=default_data_collator,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/905k [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

ViTModel LOAD REPORT from: google/vit-base-patch16-384
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/423M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

CamembertForCausalLM LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                                                                   | Status     | 
----------------------------------------------------------------------+------------+-
roberta.pooler.dense.weight                                           | UNEXPECTED | 
roberta.pooler.dense.bias                                             | UNEXPECTED | 
roberta.encoder.layer.{0...11}.crossattention.self.query.weight       | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.output.dense.bias       | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.output.dense.weight     | MISSING    | 
lm_head.decoder.weight                                                | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.output.LayerNorm.bias   | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.self.key.bias           | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.self.quer

In [6]:
trainer.train()

#trainer.train(resume_from_checkpoint=True)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 0}.


Step,Training Loss,Validation Loss
1000,22.787576,5.512244
2000,21.007810,5.155859
3000,20.571221,5.069208
4000,20.319968,5.021909
5000,20.180872,4.998560
6000,20.107711,4.971484


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [14]:
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, XLMRobertaTokenizer
import os
import glob

CHECKPOINT_PATH = "/content/drive/MyDrive/caption_model_sota/checkpoint-6000"

encoder_id = "google/vit-base-patch16-384"
decoder_id = "airesearch/wangchanberta-base-att-spm-uncased"

processor = ViTImageProcessor.from_pretrained(encoder_id)
tokenizer = XLMRobertaTokenizer.from_pretrained(decoder_id)
tokenizer.bos_token = tokenizer.cls_token
tokenizer.eos_token = tokenizer.sep_token

model = VisionEncoderDecoderModel.from_pretrained(CHECKPOINT_PATH).to("cuda")
model.eval()

gen_kwargs = {
    "max_length": 64,
    "num_beams": 4,
    "early_stopping": True,
    "length_penalty": 1.2,
    "no_repeat_ngram_size": 3,
    "bos_token_id": tokenizer.bos_token_id,
    "eos_token_id": tokenizer.eos_token_id,
    "pad_token_id": tokenizer.pad_token_id,
    "decoder_start_token_id": tokenizer.bos_token_id,
}

base_path = '/content/super-ai-engineer-ss-6-thai-language-image-captioning'
sample_sub_path = f'{base_path}/sample_submission.csv'
sample_sub = pd.read_csv(sample_sub_path, dtype={'image_id': str})

results = []
error_printed = False

with torch.no_grad():
    for img_id in tqdm(sample_sub['image_id']):
        img_name = str(img_id)
        search_pattern = f"{base_path}/**/{img_name}.*"
        found_files = glob.glob(search_pattern, recursive=True)

        if not found_files:
            if not error_printed:
                error_printed = True
            caption = "ภาพของสิ่งของบางอย่าง"
        else:
            img_path = found_files[0]
            try:
                image = Image.open(img_path).convert("RGB")
                pixel_values = processor(images=image, return_tensors="pt").pixel_values.to("cuda")
                output_ids = model.generate(pixel_values, **gen_kwargs)
                caption = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
            except Exception as e:
                if not error_printed:
                    error_printed = True
                caption = "ภาพของสิ่งของบางอย่าง"

        results.append({"image_id": img_id, "caption": caption})

submission_df = pd.DataFrame(results)
final_csv_name = "submission_FIXED_RESCUE.csv"
submission_df.to_csv(final_csv_name, index=False)


Loading weights:   0%|          | 0/524 [00:00<?, ?it/s]

100%|██████████| 2000/2000 [15:56<00:00,  2.09it/s]


In [ ]:
import torch
import pandas as pd
import glob
import os
from tqdm import tqdm
from PIL import Image

model.eval()
model.to("cuda")

corrections = {
    "สีน้ตาล": "สีน้ำตาล", "กลัง": "กำลัง", "สีด": "สีดำ",
    "แม่น้": "แม่น้ำ", "จนวน": "จำนวน", "ทท่า": "ทำท่า",
    "กแพง": "กำแพง", "น้ใส": "น้ำใส", "น้จิ้ม": "น้ำจิ้ม",
    "ถูกนมา": "ถูกนำมา", "นมาวาง": "นำมาวาง", "ในน้": "ในน้ำ",
    "ผิวน้": "ผิวน้ำ", "ริมน้": "ริมน้ำ", "น้ทะเล": "น้ำทะเล",
    "น้ตก": "น้ำตก", "น้พุ": "น้ำพุ", "น้แข็ง": "น้ำแข็ง",
    "อาบน้": "อาบน้ำ", "ว่ายน้": "ว่ายน้ำ", "ดน้": "ดำน้ำ",
    "ใต้น้": "ใต้น้ำ", "แก้วน้": "แก้วน้ำ", "ขวดน้": "ขวดน้ำ",
    "ทอาหาร": "ทำอาหาร", "จดจ": "จดจำ", "ประจ": "ประจำ",
    "บ่อน้": "บ่อน้ำ"
}

results = []
test_dir = f'{base_path}/test/test'
test_images = glob.glob(f"{test_dir}/*.jpg") + glob.glob(f"{test_dir}/*.png")

with torch.no_grad():
    for img_path in tqdm(test_images):
        raw_id = os.path.basename(img_path).split('.')[0]

        try:
            image = Image.open(img_path).convert("RGB")
            pixel_values = image_processor(image, return_tensors="pt").pixel_values.to("cuda")

            generated_ids = model.generate(
                pixel_values,
                max_length=60,
                num_beams=8,
                repetition_penalty=1.1,
                length_penalty=1.2,
                early_stopping=True,
                decoder_start_token_id=tokenizer.bos_token_id
            )

            caption = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

            clean_caption = caption.replace(" ", "").replace("<_>", "").replace(" ", "").strip()

            for wrong, right in corrections.items():
                clean_caption = clean_caption.replace(wrong, right)

            if not clean_caption:
                clean_caption = "ภาพถ่าย"

            results.append({"image_id": raw_id, "caption": clean_caption})

        except Exception as e:
            results.append({"image_id": raw_id, "caption": "ภาพถ่าย"})

df_final = pd.DataFrame(results)

df_final['image_id'] = df_final['image_id'].astype(str).str.zfill(5)
sample = pd.read_csv(f'{base_path}/sample_submission.csv')
sample['image_id'] = sample['image_id'].astype(str).str.zfill(5)

final_sub = pd.merge(sample[['image_id']], df_final, on='image_id', how='left')
final_sub['caption'] = final_sub['caption'].fillna("ภาพถ่าย")

final_name = 'submission.csv'
final_sub.to_csv(final_name, index=False)

100%|██████████| 2000/2000 [15:08<00:00,  2.20it/s]


In [ ]:
import torch
import pandas as pd
import glob
import os
from tqdm import tqdm
from PIL import Image

model.eval()
model.to("cuda")

ultimate_corrections = {
    "สีน้ตาล": "สีน้ำตาล", "สีน้้ตาล": "สีน้ำตาล", "สีด": "สีดำ", "สีดำา": "สีดำ", "ขาวด": "ขาวดำ", "สีน้เงิน": "สีน้ำเงิน",
    "กลัง": "กำลัง", "ทท่า": "ทำท่า", "ถูกนมา": "ถูกนำมา", "นมาวาง": "นำมาวาง", "ทจาก": "ทำจาก", "ทอาหาร": "ทำอาหาร",
    "จดจ": "จดจำ", "ประจ": "ประจำ", "นั่งทงาน": "นั่งทำงาน", "ทการ": "ทำการ", "ดน้": "ดำน้ำ",
    "แม่น้": "แม่น้ำ", "จนวน": "จำนวน", "กแพง": "กำแพง", "น้ใส": "น้ำใส", "น้จิ้ม": "น้ำจิ้ม",
    "ในน้": "ในน้ำ", "ผิวน้": "ผิวน้ำ", "ริมน้": "ริมน้ำ", "น้ทะเล": "น้ำทะเล", "น้ตก": "น้ำตก", "น้พุ": "น้ำพุ",
    "น้แข็ง": "น้ำแข็ง", "อาบน้": "อาบน้ำ", "ว่ายน้": "ว่ายน้ำ", "ใต้น้": "ใต้น้ำ", "แก้วน้": "แก้วน้ำ", "ขวดน้": "ขวดน้ำ",
    "บ่อน้": "บ่อน้ำ", "น้ประปา": "น้ำประปา", "น้ซอส": "น้ำซอส", "น้ซุป": "น้ำซุป", "น้พริก": "น้ำพริก", "น้อยู่": "น้ำอยู่",
    "ส้มต": "ส้มตำ", "ต้มย": "ต้มยำ", "ตถั่ว": "ตำถั่ว", "ยขนมจีน": "ยำขนมจีน", "น้ยพริก": "น้ำพริก",
    "หลายล": "หลายลำ", "ลหนึ่ง": "ลำหนึ่ง", "1ล": "1ลำ", "2ล": "2ลำ", "ลเล็ก": "ลำเล็ก", "เรือล": "เรือลำ"
}

results = []
test_dir = f'{base_path}/test/test'
test_images = glob.glob(f"{test_dir}/*.jpg") + glob.glob(f"{test_dir}/*.png")

with torch.no_grad():
    for img_path in tqdm(test_images):
        raw_id = os.path.basename(img_path).split('.')[0]

        try:
            image = Image.open(img_path).convert("RGB")
            pixel_values = image_processor(image, return_tensors="pt").pixel_values.to("cuda")

            generated_ids = model.generate(
                pixel_values,
                max_length=35,
                num_beams=4,
                repetition_penalty=1.0,
                length_penalty=1.0,
                early_stopping=True,
                decoder_start_token_id=tokenizer.bos_token_id
            )

            caption = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
            clean_caption = caption.replace(" ", "").replace(" ", "").replace("<_>", "").strip()

            for wrong, right in ultimate_corrections.items():
                clean_caption = clean_caption.replace(wrong, right)

            if not clean_caption:
                clean_caption = "ภาพถ่าย"

            results.append({"image_id": raw_id, "caption": clean_caption})

        except Exception as e:
            results.append({"image_id": raw_id, "caption": "ภาพถ่าย"})

df_final = pd.DataFrame(results)
df_final['image_id'] = df_final['image_id'].astype(str).str.zfill(5)

sample = pd.read_csv(f'{base_path}/sample_submission.csv')
sample['image_id'] = sample['image_id'].astype(str).str.zfill(5)

final_sub = pd.merge(sample[['image_id']], df_final, on='image_id', how='left')
final_sub['caption'] = final_sub['caption'].fillna("ภาพถ่าย")

final_name = 'submission.csv'
final_sub.to_csv(final_name, index=False)


100%|██████████| 2000/2000 [14:19<00:00,  2.33it/s]
